In [32]:
#Environment Imports & Settings

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn validation & preprocessing tools
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error

# Configure display aesthetics
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)


In [42]:
#Load Data and drop ID from predictions 

In [43]:
train_path='../data/challenge1_train.csv'
test_path='../data/challenge1_test.csv'

train_df=pd.read_csv(train_path)
test_df=pd.read_csv(test_path)

#print statements to test if the data was loaded properly
print(f"Training shape: {train_df.shape} (Expected: 7398 rows)")
print(f"Testing shape:  {test_df.shape} (Expected: 2467 rows)")

#As per instructed in the rubric we drop ID from predictions 
X=train_df.drop(columns=['ID','price'])    # the x is defined but out features (except target price and ID)
Y=train_df['price']   #price is out target so it is the y 

#save test IDs for final prediction.csv
test_ids=test_df['ID']
X_test=test_df.drop(columns=['ID'])

train_df.head()




Training shape: (7398, 10) (Expected: 7398 rows)
Testing shape:  (2467, 9) (Expected: 2467 rows)


,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [44]:
#INspect Data Types & missing Values 

In [45]:
# 1. Feature types and missing count
print("=== Column Types and Non-Null Counts ===")
print(X.info())

# 2. Check which columns contain null values
null_counts = X.isnull().sum()
print("\n=== Missing Values Per Column ===")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No missing values found.")

# 3. Target variable statistical summary
print("\n=== Target Variable ('price') Statistics ===")
print(y.describe())

=== Column Types and Non-Null Counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   bathrooms    7398 non-null   float64
 1   bedrooms     7398 non-null   int64  
 2   square_feet  7398 non-null   int64  
 3   latitude     7398 non-null   float64
 4   longitude    7398 non-null   float64
 5   category     7398 non-null   object 
 6   has_photo    7398 non-null   object 
 7   state        7398 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 462.5+ KB
None

=== Missing Values Per Column ===
No missing values found.

=== Target Variable ('price') Statistics ===
count    7398.000000
mean     1462.805488
std       821.583589
min       200.000000
25%       949.000000
50%      1273.500000
75%      1695.000000
max      8800.000000
Name: price, dtype: float64


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define columns explicitly based on the type of feature either numerical or cathegorical 
num_features = ['bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude']
cat_features = ['category', 'has_photo', 'state']

# Numerical Pipeline: Median imputation (safety for test set) + StandardScaler
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Mode imputation (safety) + One-Hot Encoding
# handle_unknown='ignore' prevents crashes if the test set has a new/unseen category
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

print("Preprocessor configured successfully.")

Preprocessor configured successfully.


In [47]:
#Validation Strategy 

In [48]:
from sklearn.model_selection import KFold, cross_validate
import numpy as np

# 5-Fold Cross-Validation with fixed seed for reproducibility
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_pipeline(pipeline, X_data, y_data, model_name="Model"):
    """
    Computes 5-fold CV Root Mean Squared Error (RMSE) for both Train and Validation.
    """
    cv_scores = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring='neg_mean_squared_error',
        return_train_score=True,
        n_jobs=-1
    )
    
    # neg_mean_squared_error returns negative values, so negate before taking the square root
    train_rmse = np.sqrt(-cv_scores['train_score'])
    val_rmse = np.sqrt(-cv_scores['test_score'])
    
    mean_val = val_rmse.mean()
    std_val = val_rmse.std()
    mean_train = train_rmse.mean()
    overfit_gap = mean_val - mean_train
    
    print(f"[{model_name}] Val RMSE: {mean_val:.2f} (± {std_val:.2f}) | Train RMSE: {mean_train:.2f} | Gap: {overfit_gap:.2f}")
    
    return {
        'Model': model_name,
        'Train RMSE': round(mean_train, 2),
        'Val RMSE Mean': round(mean_val, 2),
        'Val RMSE Std': round(std_val, 2),
        'Overfit Gap': round(overfit_gap, 2)
    }

In [49]:
#Baseline Models Comparassion


In [51]:
from sklearn.base import BaseEstimator, RegressorMixin

class NormalEquationLinearRegression(BaseEstimator, RegressorMixin):
    """
    Ordinary Least Squares Linear Regression solved analytically
    using the Normal Equation: theta = (X^T * X)^(-1) * X^T * y
    """
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.theta_ = None
        self.intercept_ = None
        self.coef_ = None

    def fit(self, X, y):
        # Convert input to numpy array if DataFrame or sparse matrix
        if hasattr(X, "toarray"):
            X_mat = X.toarray()
        elif hasattr(X, "values"):
            X_mat = X.values
        else:
            X_mat = np.asarray(X)
            
        y_vec = np.asarray(y).reshape(-1, 1)

        # Add bias / intercept column of ones: [1, x1, x2, ...]
        if self.fit_intercept:
            bias_col = np.ones((X_mat.shape[0], 1))
            X_b = np.hstack([bias_col, X_mat])
        else:
            X_b = X_mat

        # Solve closed-form using Moore-Penrose pseudo-inverse (np.linalg.pinv)
        # This prevents singularity/non-invertible matrix crashes from collinear dummy variables
        theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y_vec
        self.theta_ = theta.flatten()

        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_

        return self

    def predict(self, X):
        if hasattr(X, "toarray"):
            X_mat = X.toarray()
        elif hasattr(X, "values"):
            X_mat = X.values
        else:
            X_mat = np.asarray(X)

        if self.fit_intercept:
            bias_col = np.ones((X_mat.shape[0], 1))
            X_b = np.hstack([bias_col, X_mat])
        else:
            X_b = X_mat

        predictions = X_b @ self.theta_
        return predictions

In [53]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Models dictionary including both mathematical and ML linear regressions
models = {
    'Linear Regression (Normal Equation / Math)': NormalEquationLinearRegression(fit_intercept=True),
    'Linear Regression (Scikit-Learn / ML)': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    # Chain preprocessor and model together into one pipeline
    full_pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    res = evaluate_pipeline(full_pipe, X, y, model_name=name)
    results.append(res)

# Display summary table
results_summary = pd.DataFrame(results).sort_values(by='Val RMSE Mean')
results_summary

[Linear Regression (Normal Equation / Math)] Val RMSE: 526.71 (± 25.23) | Train RMSE: 520.87 | Gap: 5.85
[Linear Regression (Scikit-Learn / ML)] Val RMSE: 526.76 (± 25.27) | Train RMSE: 520.87 | Gap: 5.89
[Decision Tree] Val RMSE: 562.27 (± 65.26) | Train RMSE: 40.67 | Gap: 521.60
[Random Forest] Val RMSE: 426.51 (± 33.60) | Train RMSE: 162.97 | Gap: 263.54
[Gradient Boosting] Val RMSE: 460.36 (± 28.09) | Train RMSE: 406.71 | Gap: 53.65


,Model,Train RMSE,Val RMSE Mean,Val RMSE Std,Overfit Gap
3,Random Forest,162.97,426.51,33.60,263.54
4,Gradient Boosting,406.71,460.36,28.09,53.65
0,Linear Regression (Normal Equation / Math),520.87,526.71,25.23,5.85
1,Linear Regression (Scikit-Learn / ML),520.87,526.76,25.27,5.89
2,Decision Tree,40.67,562.27,65.26,521.60
